In [0]:
import importlib
import spotify_tokens
import requests
import json
import uuid
from datetime import datetime, timezone

# Reload the module to pick up any changes
importlib.reload(spotify_tokens)
from spotify_tokens import Tokens

def get_recently_played(access_token):
    request_url = 'https://api.spotify.com/v1/me/player/recently-played'
    headers = { 'Authorization': f"Bearer {access_token}" }
    params = { "limit": 50 }

    response = requests.get(request_url, headers=headers, params=params)
    response.raise_for_status()
    construct_data(response.json())

def construct_data(data):
    run_id = str(uuid.uuid4())
    rows = []
    records_rejected = 0
    records_read = len(data["items"])
    start_time = datetime.now(timezone.utc)
    
    for item in data["items"]:
        try:
            track = item['track']
            album = track['album']

            rows.append({
                "run_id": run_id,
                "album_id": album["id"],
                "album_name": album["name"],
                "artist_id": track["artists"][0]["id"],
                "artist_name": track["artists"][0]["name"],
                "duration_ms": track["duration_ms"],
                "track_id": track["id"],
                "track_name": track["name"],
                "played_at": item["played_at"]
            })
        except:
            records_rejected += 1
    
    write_to_db(rows)
    write_to_pipeline_table(records_read, records_rejected, run_id, len(rows), start_time)

def write_to_pipeline_table(read, rejected, run_id, written, start_time):
    end_time = datetime.now(timezone.utc)
    pipeline_status = 'success'
    if written == 0:
        pipeline_status = 'failed'
    if rejected > 0:
        pipeline_status = 'partial'

    audit_row = [{
        "run_id": run_id,
        "start_time": start_time,
        "end_time": end_time,
        "status": pipeline_status,
        "records_read": read,
        "records_rejected": rejected,
        "records_written": written
    }]

    try:
        audit_df = spark.createDataFrame(audit_row)
        audit_df.write.mode('append').saveAsTable("spotify_project.metadata.pipeline_runs")
        print("Processing data -> spotify_project.metadata.pipeline_runs")
    except:
        print("error writing to table -> spotify_project.metadata.pipeline_runs")

def write_to_db(data):
    try:
        print(f"Processing data -> spotify_project.bronze.spotify_ingest")
        df = spark.createDataFrame(data)
        df = df.write.mode('append').saveAsTable("spotify_project.bronze.spotify_ingest") 
        return True
    except:
        print("Error writing to table -> spotify_project.bronze.spotify_ingest")
        return False
    
try:
    token = Tokens()
    access_token = token.get_access_token()
    get_recently_played(access_token)
except Exception as e:
    print(f'Error fetching data or invalid access token: {type(e).__name__}: {str(e)}')